# N2 · Inverse PINN: what loads is this cantilever carrying?
**Physics-informed machine learning for civil engineers · hands-on 2 · about 30 minutes**

In N1 we knew the load and the network found the deflection. **Now we turn it around.** A steel cantilever bracket carries an access platform. It has been surveyed: we have the deflection at 16 points along it, with noise. We do **not** know what is on it. There is a uniform load `q` (stored material, snow) and a concentrated load `P` from a plant item at mid-span, and both magnitudes are unknown:

$$EI\,w''''(x) \;=\; q \;+\; P\,\delta(x-a), \qquad a = L/2, \qquad q = ?, \quad P = ?$$

The trick is simple: **each unknown load becomes one more trainable number**, trained together with the weights of the network. The beam equation links the loads to the deflection, so the readings pin them down — or so we hope.

| Part | What you do | Minutes |
|---|---|---|
| A | **TODO 1, 2, 3**: a trainable load, the data loss, the physics loss with the load inside | 12 |
| B | warm-up, one unknown: train, read off `q`, compare with a data-only network | 6 |
| C | **class experiment 2**: two unknown loads, and your birth month decides how wide the point load is | 5 |
| D | **class experiment 3**: one unknown again, and your birth month decides the loss weights | 5 |
| E | why the point load vanishes (no training: three numbers) | 4 |

> **Student version.** Fill in the cells marked `TODO` (look for the word `None`). Each TODO has an explanation, a **hint** you can open, and a **check cell** that prints `PASSED` or `NOT YET`. The training cells stop with a clear message if a TODO is not finished. The solution is in `N2_solution.ipynb`: look at it *after* you have tried.

## Set-up
Same libraries, beam and exact solution as before. In this notebook we also need the exact solution for a **point load** (part C).

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn                    # same libraries as in N0

torch.set_default_dtype(torch.float64)   # 64-bit numbers: needed for clean 4th derivatives (see N0 part 3)
print("PyTorch", torch.__version__, "| numbers are", torch.get_default_dtype())

In [ ]:
# ---- The beam, in SI units -------------------------------------------------------------------------
EI   = 1.755e7        # bending stiffness of an IPE 300, N m^2
L    = 3.0            # length of the cantilever, m
Q_SI = 10e3           # uniform load, N/m  (= 10 kN/m)
P_SI = 30e3           # point load, N (= 30 kN = q0 L): P = 1 in scaled units means exactly this

# ---- Scaling: we work with numbers of order 1 (N1 part D shows why this matters) -------------------
# x is measured in units of L, and the deflection in units of W0 = the tip deflection of THIS beam
# under the reference load, so that a correct deflection runs from 0 at the clamp to 1.0 at the tip.
W0 = Q_SI * L**4 / (8 * EI)   # unit of deflection = 5.8 mm  (the textbook tip deflection q L^4 / 8EI)
M0 = EI * W0 / L**2           # what one unit of w''  is worth = 11.25 kN m
F0 = EI * W0 / L**3           # what one unit of w''' is worth = 3.75 kN

def to_mm(w):    return w * W0 * 1e3     # scaled deflection -> millimetres
def to_kNm(M):   return M * M0 / 1e3     # scaled moment     -> kN m
def to_kN(V):    return V * F0 / 1e3     # scaled force      -> kN

# ---- Exact (textbook) solution for the uniform load, in scaled units: q = 1 ----------------------------
# Fixed end at x = 0, free end at x = 1.
def w_unif(x): return x**2 * (6 - 4*x + x**2) / 3      # deflection  w:  0 at the clamp, 1.0 at the tip
def M_unif(x): return 4 * (1 - x)**2                    # bending moment  M = w''   (to_kNm -> kN m)
def V_unif(x): return 8 * (1 - x)                       # shear force     V = -w'''  (to_kN  -> kN)

# The beam equation in these units is  w'''' = 8 q.  The 8 is the same 8 as in the tip deflection q L^4 / 8EI.
# We always divide by it and use the form  w''''/8 = q, which reads: the load the network implies = the load we assume.
GRID = np.linspace(0, 1, 201)                           # 201 points along the beam, used for plots

# ---- Two numbers to judge a fit ------------------------------------------------------------------------
def r2(y, yhat):      return 1 - np.sum((y - yhat)**2) / np.sum((y - y.mean())**2)   # R^2: 1.0 = perfect fit
def rel_l2(y, yhat):  return np.linalg.norm(y - yhat) / np.linalg.norm(y)             # relative error: 0.1 = 10 % off

# ---- Virtual sensor readings: exact deflection + random noise -----------------------------------------------
def make_sensors(n, x_first=0.05, x_last=1.0, noise=0.10, seed=0, q=1.0, P=0.0):
    """n virtual sensors between x_first and x_last. noise = standard deviation as a fraction of the tip deflection."""
    xs = np.linspace(x_first, x_last, n)
    w_exact = q * w_unif(xs)                  # exact deflection at the sensors
    w_tip = q * w_unif(1.0)                   # exact tip deflection (the noise is a fraction of this)
    if P != 0:                                # a point load as well (from N2 part C on)
        w_exact = w_exact + P * w_pt(xs)
        w_tip = w_tip + P * w_pt(1.0)
    ys = w_exact + noise * w_tip * np.random.RandomState(seed).randn(n)    # same seed = same noise every run
    return xs, ys

# ---- what those units mean, in numbers --------------------------------------------------------------
rows = [("x~ = x / L",      f"L = {L:.2f} m",                   "the free end"),
        ("w~ = w / W0",     f"W0 = qL^4/8EI = {W0 * 1e3:.2f} mm", "the tip deflection of this beam under q0"),
        ("q~ = q / q0",     f"q0 = {Q_SI / 1e3:.1f} kN/m",        "the REFERENCE load (not the answer: q is what we look for)"),
        ("P~ = P / (q0 L)", f"q0 L = {Q_SI * L / 1e3:.1f} kN",    "a point load as large as the whole uniform load"),
        ("s  = width / L",  f"s = 0.05 is {0.05 * L * 1e3:.0f} mm",   "the width of the bearing plate (part C)")]
print("The scaled variables, and what each scale is worth on our beam:\n")
for _v, _sc, _meaning in rows:
    print(f"   {_v:<18}{_sc:<27}{_meaning}")
print(f"\n   The equation becomes   w~''''/8 = q~ + P~ d(x~ - a~)   -- the 8 is the one in qL^4/8EI,")
print(f"   so w~''''/8 is the load the network implies, in units of q0. The cell below derives this.")
print(f"\n   A correct deflection runs from 0 at the clamp to {w_unif(1.0):.2f} at the tip, and the readings do too.")
print(f"   Moment and shear come back in engineering units: to_kNm(w'') gives {to_kNm(M_unif(0.0)):.0f} kN m at the clamp,")
print(f"   to_kN(-w''') gives {to_kN(V_unif(0.0)):.0f} kN at the support.   A recovered q~ of 1.25 would mean {1.25 * Q_SI / 1e3:.1f} kN/m.")

### Where the scaling comes from (and what the point load does to it)

The bracket in SI units, carrying a uniform load `q` and a point load `P` at `x = a`:

$$EI\,\frac{d^4w}{dx^4} \;=\; q \;+\; P\,\delta(x-a), \qquad 0 \le x \le L$$

**Step 1 — choose one scale for each quantity.** Every scale is built from things we know before we start: the geometry `L`, the stiffness `EI`, and a **reference load** `q0` that we nominate.

| quantity | scaled variable | its scale | so "= 1" means |
|---|---|---|---|
| position | `x̃ = x / L` | `L` | the free end |
| deflection | `w̃ = w / W₀` | `W₀ = q0 L⁴ / 8EI` | the tip deflection of this beam under `q0` |
| uniform load | `q̃ = q / q0` | `q0` | the reference load |
| point load | `P̃ = P / (q0 L)` | `q0 L` | a point load as large as the whole uniform load |
| bearing width | `s = width / L` | `L` | — |

**Step 2 — substitute.** Each derivative with respect to `x` brings out a `1/L`, so four of them bring `1/L⁴`:

$$\frac{d^4w}{dx^4} \;=\; \frac{W_0}{L^4}\,\tilde w'''' \qquad\Longrightarrow\qquad EI\,\frac{W_0}{L^4}\,\tilde w''''\;=\;q_0\,\tilde q \;+\; P\,\delta(x-a)$$

The delta function is a *density*, so it scales as well: it must integrate to 1 either way and `dx = L\,dx̃`, so `δ(x − a) = δ(x̃ − ã) / L`. With `P = P̃ q0 L` the point-load term becomes `q0 P̃ δ(x̃ − ã)`. And on the left, `EI · W₀/L⁴ = q0/8` by the choice of `W₀`.

**Step 3 — divide through by `q0`:**

$$\frac{\tilde w''''}{8} \;=\; \tilde q \;+\; \tilde P\,\delta(\tilde x-\tilde a)$$

Three things to take from it.

* **The 8 is the only constant left**, and it is the 8 of the textbook tip deflection `q L⁴/8EI`. It is there because we chose to measure `w` in units of that deflection. Read `w̃''''/8` as **the load the network implies**, in units of `q0`; the physics residual is then one load minus another.
* **The point load does not change the scaling.** It is scaled by `q0 L`, the *total* uniform load, which is why `q̃ = 1` and `P̃ = 1` are comparable sizes and why the equation keeps exactly the same form. Everything part C does with two unknown loads rests on this one line. In the code, `load = q̃ + P̃ · δ_s` and the residual is `w4 / 8 - load`.
* **Only `q0` is a choice.** `L` and `EI` are measured; `q0` is nominated by us. Choosing a different one rescales `w` and `q` together and leaves the equation identical — which is exactly why a recovered `q̃ = 1.25` has to be reported *together with* the reference it refers to.

In part C the delta is replaced by a narrow Gaussian of width `s` and unit area — a real bearing plate rather than a mathematical point. Same equation, with `δ` replaced by `δ_s`.

In [ ]:
# ---- Exact solution for a point load P = 1 (30 kN) at x = a = 1/2 (scaled units) ----------------------
def w_pt(x, a=0.5): return 8 * np.where(x <= a, x**2 * (3*a - x) / 6, a**2 * (3*x - a) / 6)   # deflection
def M_pt(x, a=0.5): return 8 * np.where(x <= a, a - x, 0.0)                                     # bending moment
def V_pt(x, a=0.5): return 8 * np.where(x <= a, 1.0, 0.0)                                       # shear (jumps at x = a)

**Plumbing.** Run the next cell once (its code is hidden: click the small `…` to look at it). It only sets up plot style, saving figures for the slides, and the **check functions** that test your TODOs. You do not need to read it.

In [ ]:
import os, json
# colours and plot style (cosmetic)
NAVY, ORANGE, GREY, TEAL, INK = "#002147", "#E9560D", "#9AA5B1", "#1B8C84", "#333F48"
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
    "font.size": 11, "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#5F6B74",
    "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": NAVY, "legend.frameon": False,
    "figure.dpi": 100, "axes.grid": True, "grid.color": "#E3E8EE", "grid.linewidth": 0.8})

# show(fig, name): tidy the layout, display the figure, and (only when run_all.py runs the notebook) save it as a slide figure
FIG_DIR = os.environ.get("PIML_FIG_DIR")
def show(fig, name=None):
    fig.tight_layout()
    if FIG_DIR and name:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()

# record(key, value) keeps a number for the slides; save_results writes them to results_N?.json (used by run_all.py)
RESULTS = {}
def record(key, value):
    RESULTS[key] = value if isinstance(value, (list, str)) else float(value)
    return value
def save_results(name):
    with open(f"results_{name}.json", "w") as f: json.dump(RESULTS, f, indent=1)

# ---- checks for the TODO cells ---------------------------------------------------------------------------------
STATUS = {}                                    # remembers which TODOs passed
def run_check(label, test, hint="", todo=None):
    """run test() and print PASSED or NOT YET (with a hint). todo = number of the TODO this check belongs to."""
    try:
        ok = bool(test())
    except Exception as e:
        ok = False
        if "NoneType" in str(e): hint = "a TODO line still says `None`: replace it with your expression (see the explanation above the cell)"
        else: hint = f"{hint}  [{type(e).__name__}: {e}]".strip()
    print(("PASSED   " if ok else "NOT YET  ") + label + ("" if ok else "\n          hint: " + hint))
    if todo is not None: STATUS[todo] = ok

def need_todos(*numbers):
    """stop with a clear message if a TODO is not finished (called before the training cells)"""
    missing = [n for n in numbers if not STATUS.get(n)]
    if missing:
        raise RuntimeError(f"TODO {missing} is not finished yet. Go back to it: its check cell must print PASSED.")
def exact_deflection(x): return x**2 * (6 - 4*x + x**2) / 3       # the exact solution, as a PyTorch formula

def check_log_q():
    """TODO 1: a trainable tensor that starts at the value we ask for"""
    run_check("TODO 1: log q is a trainable tensor", lambda: make_log_q(-1.0).requires_grad, "use requires_grad=True", todo=1)
    run_check("TODO 1: it starts at the value we ask for", lambda: abs(make_log_q(-0.5).item() + 0.5) < 1e-12, "use torch.tensor(start, ...)", todo=1)

def check_data_loss():
    """TODO 2: zero for exact readings, positive when the readings are 10 % too large"""
    xs_t, ys_t = make_sensors(8, 0.1, 1.0, noise=0.0)                          # 8 exact readings, no noise
    X_t = torch.tensor(xs_t.reshape(-1, 1))
    Y_t = torch.tensor(ys_t.reshape(-1, 1))
    run_check("TODO 2: zero for exact readings", lambda: data_loss(exact_deflection, X_t, Y_t).item() < 1e-20, todo=2)
    run_check("TODO 2: positive when the readings are 10 % too large", lambda: data_loss(exact_deflection, X_t, 1.1 * Y_t).item() > 1e-4, todo=2)

def check_physics_loss():
    """TODO 3: exact solution with the right load -> zero; with a wrong load -> not zero"""
    xc = torch.rand(50, 1).requires_grad_(True)                                # 50 random test points
    run_check("TODO 3: exact solution, right load: zero", lambda: physics_loss(exact_deflection, xc, 1.0).item() < 1e-18, todo=3)
    run_check("TODO 3: exact solution, wrong load: not zero", lambda: physics_loss(exact_deflection, xc, torch.tensor(2.0)).item() > 0.9, todo=3)

# ---- the survey, drawn and printed (cosmetic; used by "The survey" section) --------------------------------
def show_survey(xs, ys, point_load=False):
    """two panels: the bracket with its unknown loads, and the readings we actually have"""
    fig, ax = plt.subplots(1, 2, figsize=(11.2, 3.6))
    a = ax[0]                                                        # ---------------- the structure
    a.set_axis_off()
    a.set_xlim(-0.17, 1.12)
    a.set_ylim(-0.80, 1.05)
    a.add_patch(plt.Rectangle((-0.09, -0.30), 0.09, 0.60, facecolor="#D8DEE5", edgecolor=NAVY, hatch="////", lw=1.4))
    a.plot([0, 1], [0, 0], color=NAVY, lw=6, solid_capstyle="butt", zorder=3)
    a.plot([0, 1], [0.42, 0.42], color=TEAL, lw=2)
    for xa in np.linspace(0.03, 0.97, 12):
        a.annotate("", xy=(xa, 0.07), xytext=(xa, 0.41), arrowprops=dict(arrowstyle="-|>", color=TEAL, lw=1.4, mutation_scale=9))
    a.text(0.06, 0.56, "q = ?", color=TEAL, fontsize=16, fontweight="bold")
    label = "P = ?" if point_load else "P = ?   (not yet: part C adds it)"
    shade = 1.0 if point_load else 0.3                               # faint while the plant item is not on the bracket
    a.annotate("", xy=(0.5, 0.09), xytext=(0.5, 0.90), arrowprops=dict(arrowstyle="-|>", color=ORANGE, lw=3.4, mutation_scale=16, alpha=shade))
    a.text(0.54, 0.90, label, color=ORANGE if point_load else GREY, fontsize=14, fontweight="bold", va="top")
    a.plot(xs, -0.19 * np.ones_like(xs), "o", color=NAVY, ms=5, zorder=4)
    for xa in xs:
        a.plot([xa, xa], [-0.03, -0.15], color=GREY, lw=0.8, ls=":")
    a.text(0.5, -0.33, f"{len(xs)} surveyed points", color=NAVY, ha="center", va="top", fontsize=11, fontweight="bold")
    a.text(0.5, -0.56, f"L = {L:.1f} m,  EI = {EI / 1e6:.1f} × 10⁶ N·m²,  both known", ha="center", va="top", fontsize=10)
    a.set_title("What we know, and what we do not")

    b = ax[1]                                                        # ---------------- the readings
    truth = w_unif(GRID) + (w_pt(GRID) if point_load else 0.0)
    exact_at = w_unif(xs) + (w_pt(xs) if point_load else 0.0)
    for x, y, e in zip(xs, ys, exact_at):
        b.plot([x * L, x * L], [to_mm(e), to_mm(y)], color=GREY, lw=0.9)
    b.plot(GRID * L, to_mm(truth), color=GREY, lw=2, ls="--", label="the truth (we would not have this)")
    b.plot(xs * L, to_mm(ys), "o", color=ORANGE, ms=7, label="the readings we have")
    b.set(xlabel="x (m)", ylabel="deflection (mm)", title="What the survey gives us")
    b.legend(fontsize=9.5, loc="upper left")
    show(fig)

def print_survey(xs, ys, point_load=False):
    """the same readings as numbers, scaled and in millimetres"""
    exact_at = w_unif(xs) + (w_pt(xs) if point_load else 0.0)
    tip = w_unif(1.0) + (w_pt(1.0) if point_load else 0.0)
    print(f"{'sensor':>7}{'x (m)':>8}{'x scaled':>10}{'reading (mm)':>14}{'reading scaled':>16}{'truth (mm)':>12}")
    for i, (x, y, e) in enumerate(zip(xs, ys, exact_at), start=1):
        print(f"{i:>7}{x * L:>8.2f}{x:>10.3f}{to_mm(y):>14.2f}{y:>16.4f}{to_mm(e):>12.2f}")
    print(f"\nnoise on each reading: 10 % of the tip deflection = {to_mm(0.10 * tip):.2f} mm")
    print("the two middle columns are the same measurement in different units; the network sees the scaled one")

---
## The survey: what we actually have
Before writing any code, look at the data. The next cell makes the survey and shows it three ways: a sketch of the bracket, a plot of the readings, and the sixteen numbers themselves.

Two things to notice.

* **The dashed line is the truth.** It is drawn here because this is a teaching notebook and we invented the readings. On a real bracket you would have the orange dots and nothing else — no curve to compare with, and no way to tell a good answer from a bad one except by checking it against statics.
* **The readings are deflections only.** No moment, no shear, no strain. Everything the PINN reports about the loads has to come out of these sixteen numbers plus the beam equation.

In [ ]:
xs, ys = make_sensors(16, x_first=0.05, x_last=1.0, noise=0.10, seed=0)   # the survey: 16 readings with 10 % noise

show_survey(xs, ys)        # the bracket, and the readings
print_survey(xs, ys)       # the same readings as numbers

## A · The three ingredients
**Given from N1** (unchanged): the network `make_net`, the derivative helper `d`, `derivs` (network output and its first four derivatives), `bc_loss` (the four boundary conditions) and `evaluate` (network on the plot grid). Read them quickly; the new pieces come after.

In [ ]:
def make_net(width=30, act=nn.Tanh):
    return nn.Sequential(nn.Linear(1, width), act(), nn.Linear(width, width), act(), nn.Linear(width, 1))

def d(y, x):
    """derivative of y with respect to x at every point (can be differentiated again)"""
    return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True)[0]

def derivs(model, x):
    """network output w and its first four derivatives with respect to x"""
    w = model(x)
    w1 = d(w, x)
    w2 = d(w1, x)
    w3 = d(w2, x)
    w4 = d(w3, x)
    return w, w1, w2, w3, w4

In [ ]:
def bc_loss(model):
    """the four boundary conditions, squared and summed (this was TODO 3 in N1)"""
    xb = torch.tensor([[0.0], [1.0]], requires_grad=True)          # row 0: fixed end, row 1: free end
    w, w1, w2, w3, _ = derivs(model, xb)
    # Each residual is divided by 8 before squaring, for the same reason as w''''/8 in the physics loss:
    # these are derivatives of w, and the 8 is what the scaling put into them. Each term is then a fractional error.
    return (w[0, 0] / 8)**2 + (w1[0, 0] / 8)**2 + (w2[1, 0] / 8)**2 + (w3[1, 0] / 8)**2

def evaluate(model, xg=GRID):
    """w, w', w'', w''' of the network on a grid, as numpy arrays"""
    x = torch.tensor(xg.reshape(-1, 1), requires_grad=True)
    w, w1, w2, w3, _ = derivs(model, x)
    return [t.detach().numpy().ravel() for t in (w, w1, w2, w3)]

def gauss(x, s, a=0.5):
    """narrow Gaussian of unit area, centred at x = a: a point load 'smeared' over a width s (used in part C)"""
    return torch.exp(-(x - a)**2 / (2 * s**2)) / (s * np.sqrt(2 * np.pi))

### TODO 1 · A trainable load (one line)
**What to do.** Complete `make_log_q`: return a **trainable** scalar tensor that holds `log q`, starting at the value `start`.

**Why.** To find `q`, the optimiser must be allowed to change it, just like it changes the weights of the network. In PyTorch, a tensor created with `requires_grad=True` and handed to an optimiser is trainable. We train `log q` instead of `q` itself because `q = exp(log q)` is **always positive**, which a load should be. We start at `log q = −1`, i.e. `q = 0.37`, deliberately *not* at the true value 1.

**How.** Create the tensor with `torch.tensor(start, requires_grad=True)`.

<details><summary>Hint (open if stuck)</summary>

`return torch.tensor(start, requires_grad=True)`
</details>

In [ ]:
def make_log_q(start=-1.0):
    """a trainable number holding log(q)"""
    return None                   # TODO 1: a torch tensor holding `start`, with requires_grad=True

In [ ]:
check_log_q()

### TODO 2 · The data loss (one line)
**What to do.** In `data_loss`, compute the mean squared misfit between the network at the sensors and the readings, **divided by the largest reading**.

**Why divide, when the set-up cell already scaled everything?** Because those are two different operations, done for two different reasons. It is worth separating them, and the slides do the same.

| | what it does | why |
|---|---|---|
| **1 · before the network** | non-dimensionalises the *problem*: x in units of L, w in units of the tip deflection W₀ = 5.77 mm | so the network reads and writes numbers of order 1. In SI units the physics term starts at 3 × 10¹² and the beam trains to the wrong sign (N1 part D) |
| **2 · here, inside the loss** | normalises *this term*, so that it is the square of a fractional error | so that the three terms are the same kind of number and the weights 0.4 / 0.4 / 0.2 mean what they look like |

Why the second is needed even after the first: the physics term is `(w''''/8 − q)²`, a difference between two **loads**, both in units of the reference load — so it is already a fractional error. The data term is a difference between two **deflections**. Dividing it by the peak reading turns it into a fractional error as well, whatever the true load happens to be: if the real load were three times our reference, the readings would be three times larger and this term nine times larger, without anything having got worse.

A useful side effect: the number becomes readable. A data loss of `0.01` is a 10 % misfit, which is exactly the noise in our readings. So a data loss resting at 0.01 has reached the noise and should go no lower; one far below it is fitting the noise. We use that in Block 8.

We use `Y.max()`, the largest **reading** — not the true tip deflection, which we would not have.

**How.** `model(X)` is the network's prediction at the sensor positions `X`; `Y` holds the readings. Take `(model(X) − Y) / Y.max()`, square it, and average with `.mean()`.

<details><summary>Hint (open if stuck)</summary>

`return (((model(X) - Y) / Y.max()) ** 2).mean()`
</details>

In [ ]:
def data_loss(model, X, Y):
    """misfit between the network at the sensors X and the readings Y, divided by the largest reading"""
    return None                   # TODO 2: mean of ((model(X) - Y) / Y.max())^2

In [ ]:
check_data_loss()

### TODO 3 · The physics loss with the load inside (two lines)
**What to do.** Same as TODO 2 in N1, but the load is now an **input** `load` that carries gradients: return the mean of `(w''''/8 − load)²`.

**Why.** Because `load` depends on the trainable `log q`, `loss.backward()` now also computes how the physics loss changes with `q`, and the optimiser moves `q` to make the equation hold. Nothing else changes: this is how the unknown enters the PINN.

**How.** `residual = w4 − load`, then square and take the mean.

<details><summary>Hint (open if stuck)</summary>

`residual = w4 / 8 - load` and `return (residual ** 2).mean()`
</details>

In [ ]:
def physics_loss(model, x, load):
    """mean squared residual of  w''''/8 = load  at the collocation points. `load` is a tensor that carries gradients."""
    w, w1, w2, w3, w4 = derivs(model, x)
    residual = None               # TODO 3a: the load the network implies (w''''/8) minus the load
    return None                   # TODO 3b: mean of the squared residual

In [ ]:
check_physics_loss()

### The training loop (given)
Read the comments. Two things are new compared with N1:

- **two optimisers**: one for the network weights (learning rate 5e-3) and one for the unknown loads (2e-2: a single number has to travel a long way in a few thousand steps);
- the load is rebuilt at every iteration from the trainable `log q`: `q = exp(log q)`. With `point_load=True` the load becomes `q + P · Gaussian(x; ½, s)`, with two trainable numbers (part C).

The total loss is `0.4 · data + 0.4 · physics + 0.2 · boundary`. **The losses and the recovered load are printed live**, every 200 iterations: watch `q` travel from the starting value 0.37 towards the truth. The function also returns a history (the load and the three losses every 200 iterations), which we plot later.

In [ ]:
def make_load(log_params, x, point_load=False, s=0.05):
    """the load seen by the physics at the collocation points x: q, plus (optionally) P times a narrow Gaussian at mid-span"""
    load = torch.exp(log_params[0])                              # q = exp(log q): always positive
    if point_load:
        load = load + torch.exp(log_params[1]) * gauss(x, s)     # + P * (point load smeared over width s)
    return load

In [ ]:
def train_inverse(xs, ys, its=4000, lr_net=5e-3, lr_load=2e-2, seed=0, start=-1.0, n_col=100,
                  wts=(0.4, 0.4, 0.2), point_load=False, s=0.05, print_every=200, verbose=True):
    """Train network AND unknown load(s) on the readings (xs, ys). Returns the network and a history: rows [iteration, q (, P), data loss, physics loss, boundary loss]."""
    need_todos(1, 2, 3)                                          # safety net: stops if a TODO is not finished
    torch.manual_seed(seed)
    model = make_net()
    log_params = [make_log_q(start)]                             # log q ...
    if point_load:
        log_params.append(make_log_q(start))                     # ... and log P
    opt_net = torch.optim.Adam(model.parameters(), lr=lr_net)    # optimiser 1: the network weights
    opt_load = torch.optim.Adam(log_params, lr=lr_load)          # optimiser 2: the unknown loads
    X = torch.tensor(xs.reshape(-1, 1))
    Y = torch.tensor(ys.reshape(-1, 1))
    history = []
    if verbose:
        print(f"{'Iteration':>10} | {'Total loss':>12} | {'Data loss':>10} | {'Physics loss':>12} | recovered load")
        print("-" * 70)
    for it in range(its):
        x = torch.rand(n_col, 1, requires_grad=True)             # new random collocation points
        load = make_load(log_params, x, point_load, s)
        L_data = data_loss(model, X, Y)
        L_phys = physics_loss(model, x, load)
        L_bc = bc_loss(model)
        loss = wts[0] * L_data + wts[1] * L_phys + wts[2] * L_bc
        opt_net.zero_grad()
        opt_load.zero_grad()
        loss.backward()
        opt_net.step()                                           # both optimisers take their step
        opt_load.step()
        if it % print_every == 0 or it == its - 1:
            loads = [torch.exp(p).item() for p in log_params]    # q (and P) = exp(log q), exp(log P)
            history.append([it] + loads + [L_data.item(), L_phys.item(), L_bc.item()])
            if verbose:
                text = f"q = {loads[0]:.3f}"
                if point_load:
                    text += f"   P = {loads[1]:.3f}"
                print(f"{it:10d} | {loss.item():12.3e} | {L_data.item():10.2e} | {L_phys.item():12.2e} | {text}", flush=True)
    return model, np.array(history)

Two reference tools for comparison: a data-only network (as in N0), and **ordinary least squares**, which fits `q` directly with the exact beam formula `w = q · w_unif(x)`.

In [ ]:
def train_data_only(xs, ys, its=3000, lr=5e-3, seed=0):
    """network trained on the readings only (as in N0)"""
    torch.manual_seed(seed)
    model = make_net()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    X, Y = torch.tensor(xs.reshape(-1, 1)), torch.tensor(ys.reshape(-1, 1))
    for it in range(its):
        loss = ((model(X) - Y) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model

def q_least_squares(xs, ys):
    """best q when the exact beam formula is used directly: w(x) = q * w_unif(x)"""
    a = w_unif(xs)
    return float(a @ ys / (a @ a))

## B · Warm-up: one unknown load
The survey above was done **before the plant item was craned on**, so there is no point load and only `q` is unknown. We hand those same 16 readings to the PINN. Training takes about a minute.

In [ ]:
t0 = time.time()
model, hist = train_inverse(xs, ys, its=4000, seed=0)                 # the readings from "The survey" above
q_hat = hist[-1, 1]                                                   # the recovered load (scaled): q in the last row of the history
w, w1, w2, w3 = evaluate(model)
R2 = r2(w_unif(GRID), w)
print(f"\nTrained in {time.time()-t0:.0f} s")
print(f"recovered load:  q = {q_hat:.3f}   (true 1.000)  ->  {q_hat * Q_SI / 1e3:.1f} kN/m against 10.0 kN/m")
print(f"plain least squares on the same readings: {q_least_squares(xs, ys):.3f}")
print(f"deflection of the PINN against the exact beam: R2 = {R2:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.3))
ax.plot(hist[:, 0], hist[:, 1], color=ORANGE, lw=2.2, label="recovered q")
ax.axhline(1.0, color=NAVY, lw=1.8, ls="--", label="true q = 1")
ax.axhline(np.exp(-1.0), color=GREY, lw=1, ls=":")
ax.text(3950, np.exp(-1.0) + 0.02, "start: q = 0.37", color=GREY, ha="right", fontsize=9.5)
ax.set(xlabel="iteration", ylabel="q (scaled)", title="The load is found while the network trains", ylim=(0, 1.3))
ax.legend(loc="lower right")
show(fig, "n2_qhist")

**Same readings, data-only network.** Same sensors, same noise, same network and same random start. The physics term is the only difference. We compare the **fixed-end moment** (exact: 45 kN·m), which no sensor measures.

In [ ]:
model_do = train_data_only(xs, ys, its=3000, seed=0)
wd, _, w2d, _ = evaluate(model_do)
M0_pinn, M0_do = w2[0], w2d[0]                                         # moment at the fixed end: first grid point of w''
print(f"fixed-end moment (exact 45 kN m):   PINN {to_kNm(M0_pinn):.1f} kN m      data-only {to_kNm(M0_do):.1f} kN m")
print(f"relative L2 error of the moment along the beam:   PINN {100*rel_l2(M_unif(GRID), w2):.0f} %     data-only {100*rel_l2(M_unif(GRID), w2d):.0f} %")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
xm = GRID * L
ax[0].plot(xm, to_mm(w_unif(GRID)), color=NAVY, lw=2.4, label="exact")
ax[0].plot(xm, to_mm(wd), color=ORANGE, lw=2, ls="--", label="data-only")
ax[0].plot(xm, to_mm(w), color=TEAL, lw=2, label=f"inverse PINN (q̂ = {q_hat:.2f})")
ax[0].plot(xs * L, to_mm(ys), "o", color=INK, ms=4.5, label="16 readings")
ax[0].set(xlabel="x (m)", ylabel="deflection (mm)", title="Deflection")
ax[0].legend(loc="upper left", fontsize=9.5)
ax[1].plot(xm, to_kNm(M_unif(GRID)), color=NAVY, lw=2.4)
ax[1].plot(xm, to_kNm(w2d), color=ORANGE, lw=2, ls="--")
ax[1].plot(xm, to_kNm(w2), color=TEAL, lw=2)
ax[1].set(xlabel="x (m)", ylabel="bending moment (kN·m)", title=f"Moment at the fixed end: PINN {to_kNm(M0_pinn):.0f}, data-only {to_kNm(M0_do):.0f}, exact 45")
show(fig, "n2_preview")

---
## C · Class experiment 2: two unknown loads, and how wide is the point load?
*(one training, about 40 seconds, then three numbers into the class form)*

Now the plant item is on the bracket: **two unknown loads**, `q` and `P`, from 16 noisy readings of the deflection.

**First, a modelling decision.** A point load is an idealisation. In the equation it is a Dirac delta,

$$EI\,w''''(x) = q + P\,\delta(x-a),$$

and `δ` is infinitely tall and infinitely thin. A neural network is a smooth function; it cannot produce that. So we replace the delta by a narrow Gaussian carrying the same total force, spread over a width `s`:

$$\delta(x-a) \;\approx\; \delta_s(x) = \frac{1}{s\sqrt{2\pi}}\exp\!\left(-\frac{(x-a)^2}{2s^2}\right), \qquad \int_0^L \delta_s\,dx = 1 .$$

This is not only a numerical trick: a real "point" load acts over a bearing area — a baseplate, a wheel print, a stillage foot. **How wide should it be? That is the experiment.** Your birth month decides the width of your plate:

| Birth month | Jan | Feb | Mar | Apr | May | Jun | Jul | Aug | Sep | Oct | Nov | Dec |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| width `s` | 0.16 | 0.14 | 0.12 | 0.10 | 0.08 | 0.06 | **0.05** | 0.04 | 0.03 | 0.02 | 0.015 | 0.01 |
| on our 3 m beam (mm) | 480 | 420 | 360 | 300 | 240 | 180 | **150** | 120 | 90 | 60 | 45 | 30 |
| peak of the load | 3.5 | 3.8 | 4.3 | 5.0 | 6.0 | 7.6 | **9.0** | 11.0 | 14.3 | 20.9 | 27.6 | 40.9 |

The last row is the height of the peak the network must build in `w''''/8`. With the uniform load alone the load is simply 1 everywhere.

**Everything else is the same for everybody**: the total force (30 kN), the sixteen sensor positions, the noise, the network and its random start. The readings are practically the same too: between the widest and the narrowest plate the deflection changes by less than 3 % of the tip movement, and the noise is 10 %. So the survey cannot tell these twelve beams apart — only the network can.

**What to do.** (1) Type your birth month below and run the cell. (2) It prints three numbers. (3) Type them into the class form (experiment 2).

**Before you run: predict.** The truth is `q = 1` and `P = 1`. Which of the two loads do you expect to survive a narrow plate — the one spread along the beam, or the concentrated one?


In [ ]:
# ---- given: the beam when the point load is spread over a width s --------------------------------------
def cumulative_integral(f, h):
    """running integral of f from the first grid point to every grid point (trapezoid rule, spacing h)"""
    return np.concatenate([[0.0], np.cumsum(0.5 * (f[1:] + f[:-1]) * h)])

def w_smeared_exact(s, a=0.5, n=4001):
    """exact deflection for q = 1 and P = 1, with P spread over a Gaussian of width s (integrate the load four times)"""
    x = np.linspace(0, 1, n)
    h = x[1] - x[0]
    load = 1 + np.exp(-(x - a)**2 / (2 * s**2)) / (s * np.sqrt(2 * np.pi))     # the load: w''''/8 = q + P * delta_s
    shear = cumulative_integral(load, h)
    shear = shear[-1] - shear                                                  # V(x) = the load to the right of x
    moment = cumulative_integral(shear, h)
    moment = moment[-1] - moment                                               # M(x) = integral of V from x to the tip
    w = 8 * cumulative_integral(cumulative_integral(moment, h), h)             # w = double integral of M, x8 for our units
    return np.interp(GRID, x, w)

def make_sensors_spread(n, s, noise=0.10, seed=0):
    """n readings from a beam carrying q = 1 and a point load P = 1 spread over a width s"""
    xs = np.linspace(0.05, 1.0, n)
    w_exact = np.interp(xs, GRID, w_smeared_exact(s))
    ys = w_exact + noise * w_exact[-1] * np.random.RandomState(seed).randn(n)
    return xs, ys

In [ ]:
birth_month = 7        # <- change me: your birth month (1 = January ... 12 = December)

S_BY_MONTH = {1: 0.16, 2: 0.14, 3: 0.12, 4: 0.10, 5: 0.08, 6: 0.06, 7: 0.05, 8: 0.04, 9: 0.03, 10: 0.02, 11: 0.015, 12: 0.01}
s_mine = S_BY_MONTH[birth_month]                                   # the width of YOUR bearing plate, as a fraction of the span
print(f"month {birth_month}  ->  s = {s_mine:g}, that is a plate {s_mine * L * 1000:.0f} mm wide on our 3 m beam")
print(f"peak of the load the network must build: {1 + 1 / (s_mine * np.sqrt(2 * np.pi)):.1f}\n")

xs_c, ys_c = make_sensors_spread(16, s_mine, noise=0.10, seed=0)   # 16 readings from a beam with this load (same noise for everybody)
show_survey(xs_c, ys_c, point_load=True)                           # the same picture as in part B, now with the plant item on
model_c, hist_c = train_inverse(xs_c, ys_c, its=4000, seed=0, point_load=True, s=s_mine, print_every=500)

q_c, P_c = hist_c[-1, 1], hist_c[-1, 2]                            # the two recovered loads (true value: 1.00 each)
err_q = 100 * abs(q_c - 1.0)                                       # error in the uniform load (%)
err_P = 100 * abs(P_c - 1.0)                                       # error in the point load (%)

print(f"\nrecovered:  q = {q_c:.2f}  (true 1.00)      P = {P_c:.2f}  (true 1.00)")
print(f"in engineering units:  q = {q_c * Q_SI / 1e3:.1f} kN/m against 10.0,   P = {P_c * Q_SI * L / 1e3:.1f} kN against 30.0")
print("\n" + "=" * 62)
print("  Type these three numbers into the class form (experiment 2):")
print(f"     birth month                     {birth_month}")
print(f"     error in the uniform load q     {err_q:.0f} %")
print(f"     error in the point load P       {err_P:.0f} %")
print("=" * 62)


In [ ]:
required = 1 + gauss(torch.tensor(GRID.reshape(-1, 1)), s_mine).numpy().ravel()      # the load the network must build
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.6))
ax[0].plot(GRID * L, required, color=NAVY, lw=2.2)
ax[0].set(xlabel="x (m)", ylabel="load  (= required w''''/8)", title=f"Your load: {s_mine * L * 1000:.0f} mm wide, peak {required.max():.0f}")
ax[1].plot(hist_c[:, 0], hist_c[:, 1], color=ORANGE, lw=2.2, label="recovered q")
ax[1].plot(hist_c[:, 0], hist_c[:, 2], color=TEAL, lw=2.2, label="recovered P")
ax[1].axhline(1.0, color=NAVY, lw=1.8, ls="--", label="truth: q = 1 and P = 1")
ax[1].set(xlabel="iteration", ylabel="load (scaled)", title=f"q = {q_c:.2f}  ({err_q:.0f} % out),   P = {P_c:.2f}  ({err_P:.0f} % out)")
ax[1].legend(fontsize=9.5)
show(fig)

**Look at your own two panels while the class plot fills.**

1. On the left is the load the network was asked to produce in its fourth derivative. How tall is your peak?
2. On the right, follow `P` through the training. Did it move towards 1, or towards 0? And what did `q` do while that happened?
3. The sixteen readings are reproduced well in every run — that is not where the failure shows. It shows in the two loads.

The class plot puts the twelve widths side by side: the error in each load against the width of the plate. Keep it in mind — Block 8 opens with it and gives the reason a name (**spectral bias**). Part E below is the first half of the explanation.


---
## D · Class experiment 3: who chose the weights?
*(one training, about 40 seconds, then three numbers into the class form)*

Back to the problem of part B: **one** unknown load `q`, the same sixteen readings, the same network, the same random start. The only thing that changes is how the three terms of the loss are weighted:

`loss = w_data × data + w_physics × physics + w_boundary × boundary`,  with `w_data + w_physics + w_boundary = 1`

Part B used `0.4 / 0.4 / 0.2`. **Who chose that?** We did, by hand. Writing the weights so that they add up to one makes the question sharp: what matters is how the loss is **shared** between the measurements, the equation and the supports — not how big the total is. (With Adam, multiplying every term by the same constant changes very little.)

Here the physics term is doing the job that a regularisation term does in classical inversion: it is the prior knowledge that stops the curve chasing the noise. `w_physics` is the strength of that regularisation.

We hold `w_boundary` at 0.2 throughout — the supports are not negotiable — and your birth month decides how the remaining 0.8 is shared between the data and the physics.

| Birth month | Jan | Feb | Mar | Apr | May | Jun | Jul | Aug | Sep | Oct | Nov | Dec |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| `w_data` | 0.001 | 0.002 | 0.008 | 0.023 | 0.073 | 0.185 | **0.4** | 0.6 | 0.727 | 0.774 | 0.792 | 0.799 |
| `w_physics` | 0.799 | 0.798 | 0.792 | 0.777 | 0.727 | 0.615 | **0.4** | 0.2 | 0.073 | 0.026 | 0.008 | 0.001 |

(`w_boundary` = 0.2 in every column, so each column adds up to 1. July is exactly part B.)

**What to do.** (1) Type your birth month below and run the cell. (2) It prints three numbers. (3) Type them into the class form (experiment 3).

**Before you run: predict.** With a large data weight the curve follows the noisy readings closely. Is the recovered load then better, or worse?

In [ ]:
birth_month = 7        # <- change me: your birth month again (1 = January ... 12 = December)

RATIO_BY_MONTH = {1: 0.001, 2: 0.003, 3: 0.01, 4: 0.03, 5: 0.1, 6: 0.3, 7: 1.0, 8: 3.0, 9: 10.0, 10: 30.0, 11: 100.0, 12: 1000.0}
ratio = RATIO_BY_MONTH[birth_month]                  # how much more the data count than the physics
w_bc = 0.2                                           # the supports are not negotiable
w_data = 0.8 * ratio / (1 + ratio)                   # the data and the physics share what is left
w_phys_i = 0.8 / (1 + ratio)
print(f"month {birth_month}  ->  w_data {w_data:.3f}  +  w_physics {w_phys_i:.3f}  +  w_boundary {w_bc:.3f}  =  {w_data + w_phys_i + w_bc:.3f}\n")

# the SAME readings as part B (xs, ys), the same network and the same random start: only the weights change
model_r, hist_r = train_inverse(xs, ys, its=4000, seed=0, wts=(w_data, w_phys_i, w_bc), print_every=500)

q_r = hist_r[-1, 1]                                  # the recovered load (true value: 1.00)
err_q_r = 100 * abs(q_r - 1.0)                       # error in the recovered load (%)
misfit = np.interp(xs, GRID, evaluate(model_r)[0]) - ys          # PINN minus reading, at the 16 sensors
rms_r = 100 * np.sqrt(np.mean(misfit ** 2)) / w_unif(1.0)        # misfit at the sensors, in % of the tip deflection

print(f"\nrecovered load  q = {q_r:.3f}  (true 1.000)   ->  {q_r * Q_SI / 1e3:.1f} kN/m against 10.0 kN/m")
print(f"misfit at the sensors: {rms_r:.1f} % of the tip deflection      (the noise in the readings is 10 %)")
print("\n" + "=" * 62)
print("  Type these three numbers into the class form (experiment 3):")
print(f"     birth month                     {birth_month}")
print(f"     error in the load q             {err_q_r:.0f} %")
print(f"     misfit at the sensors           {rms_r:.1f} %")
print("=" * 62)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.6))
ax[0].plot(hist_r[:, 0], hist_r[:, 1], color=ORANGE, lw=2.2, label="recovered q")
ax[0].axhline(1.0, color=NAVY, lw=1.8, ls="--", label="true q = 1")
ax[0].set(xlabel="iteration", ylabel="q (scaled)", title=f"w_data {w_data:.3f} / w_physics {w_phys_i:.3f}: q = {q_r:.2f}")
ax[0].legend(fontsize=9.5)
ax[1].semilogy(hist_r[:, 0], hist_r[:, -3], color=TEAL, lw=2, label="data loss")
ax[1].semilogy(hist_r[:, 0], hist_r[:, -2], color=NAVY, lw=2, label="physics loss")
ax[1].semilogy(hist_r[:, 0], hist_r[:, -1], color=ORANGE, lw=2, label="boundary loss")
ax[1].set(xlabel="iteration", ylabel="loss (not weighted)", title="Which term stops falling?")
ax[1].legend(fontsize=9.5)
show(fig)

**Two things to look at in your own run.**

1. Your misfit at the sensors against the 10 % noise in the readings. A misfit far **below** 10 % means the curve is following the noise rather than the beam — over-fitting, which is what a large data weight buys. Far above it means the readings are being ignored.
2. The right-hand panel: with your weights, which of the three terms stops falling first?

**And the question the class plot will raise.** Everybody had the same sixteen readings. On a real structure the true load is unknown, so every one of us would report a different load and each could defend it. The weights are part of the model, and they belong in the report with the answer.

Extra 3 in the solution notebook lets the training choose the weights instead (ReLoBRaLo), which is where Block 8 goes next.

---
## E · Why does the point load vanish?
*(no training: three numbers)*

If your run in part C lost `P`, here is why. Two things work together, and the next cell computes the numbers behind them for the July plate, `s = 0.05`.

1. **The physics loss is cheapest with `P = 0`.** The residual is `w''''/8 − q − P · δ_s`. A smooth network cannot build the narrow peak quickly (this is *spectral bias*; Block 8 and N3 return to it). While the peak is missing, the physics loss is about `P²` times the "peak energy" of `δ_s`, which is large. The optimiser has two ways to bring it down: teach the network a spike (slow, a thousand weights) or shrink the single number `P` (fast, and the load has the larger learning rate). It shrinks `P`.
2. **The data do not object.** On these sixteen sensors the deflection caused by `P` has almost the same shape as the deflection caused by `q`. A larger `q` alone reproduces the readings well inside the noise. So the data loss stays small while `P` disappears, and `q` rises to make up for it.

And once `P` is small it stays small: we train `log P`, and the gradient with respect to `log P` is `P` times the gradient with respect to `P`, so it fades as `P` shrinks.

In [ ]:
# 1. the "peak energy": the physics loss per unit P^2 if the network ignores the peak and only matches its average
xf = np.linspace(0, 1, 20001)
g = np.exp(-(xf - 0.5) ** 2 / (2 * 0.05 ** 2)) / (0.05 * np.sqrt(2 * np.pi))       # the smeared point load, s = 0.05
print(f"1. smeared load: peak {g.max():.1f}, average {g.mean():.2f};  physics loss if the peak is ignored = {np.mean((g - g.mean()) ** 2):.1f} x P^2")
print("   (a healthy physics loss in this notebook is about 0.0001)")

# 2. how well can q ALONE reproduce the exact readings of a beam with q = 1 and P = 1?
xs_e = np.linspace(0.05, 1.0, 16)                    # the sensor positions
y_true = w_unif(xs_e) + w_pt(xs_e)                   # exact deflection for q = 1, P = 1
q_alone = float(w_unif(xs_e) @ y_true / (w_unif(xs_e) @ w_unif(xs_e)))          # best fit with q only
worst = 100 * np.abs(y_true - q_alone * w_unif(xs_e)).max() / y_true[-1]
print(f"2. q alone = {q_alone:.2f} reproduces the readings to within {worst:.1f} % of the tip deflection (the noise is 10 %)")

# 3. how similar are the two deflection shapes at the sensors?
print(f"3. correlation between the deflection shapes of q and of P at the sensors: {np.corrcoef(w_unif(xs_e), w_pt(xs_e))[0, 1]:.4f}")

Read the three numbers: ignoring the peak costs a physics loss of about `4.6 · P²`, tens of thousands of times a healthy value; `q ≈ 1.86` alone fits the readings to within 1.7 % of the tip deflection, far below the 10 % noise; and the two shapes are 99.9 % correlated. Compare `1.86` with the `q` your inverse run found.

**So there are two different problems here**, and they need different remedies:

- the network cannot represent a sharp load → N3 Lab 3 removes the spike with **domain decomposition**;
- the sensors cannot separate `q` from `P` → no algorithm can fix that; N3 Lab 4 shows which **extra gauges** can.

Both are taken up in Block 8.

---
### After the course
The solution notebook has three further experiments, a few minutes each:

* **Extra 1** — how the number of sensors and the noise level change the recovered load.
* **Extra 2** — the two-unknown problem at the reference width, repeated over four noise draws, against ordinary least squares.
* **Extra 3** — adaptive loss weights (ReLoBRaLo) instead of the hand-chosen ones of class experiment 3.